In [0]:
DROP TABLE IF EXISTS catalog.default.customers_masked;

CREATE TABLE catalog.default.customers_masked (
  id INT,
  region STRING,
  email STRING,
  ssn STRING,
  amount DECIMAL(10,2)
)
USING DELTA;

INSERT INTO catalog.default.customers_masked VALUES
  (1, 'EU', 'alice@example.com', '111-11-1111', 100.00),
  (2, 'US', 'bob@example.com',   '222-22-2222', 200.00),
  (3, 'EU', 'carol@example.com', '333-33-3333', 300.00);

In [0]:
DROP TABLE IF EXISTS catalog_commits.default.customers_masked;

CREATE TABLE catalog_commits.default.customers_masked (
  id INT,
  region STRING,
  email STRING,
  ssn STRING,
  amount DECIMAL(10,2)
)
USING DELTA
TBLPROPERTIES (
  'delta.feature.catalogManaged' = 'supported'
);

INSERT INTO catalog_commits.default.customers_masked VALUES
  (1, 'EU', 'alice@example.com', '111-11-1111', 100.00),
  (2, 'US', 'bob@example.com',   '222-22-2222', 200.00),
  (3, 'EU', 'carol@example.com', '333-33-3333', 300.00);

In [0]:
CREATE OR REPLACE FUNCTION catalog.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).+(@.*$)', '$1***$2');

CREATE OR REPLACE FUNCTION catalog.default.mask_ssn(ssn STRING)
RETURNS STRING
RETURN '***-**-****';


CREATE OR REPLACE FUNCTION catalog_commits.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).+(@.*$)', '$1***$2');

CREATE OR REPLACE FUNCTION catalog_commits.default.mask_ssn(ssn STRING)
RETURNS STRING
RETURN '***-**-****';

In [0]:
CREATE OR REPLACE FUNCTION catalog.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).+(@.*$)', '$1***$2');

CREATE OR REPLACE FUNCTION catalog.default.mask_ssn(ssn STRING)
RETURNS STRING
RETURN '***-**-****';


CREATE OR REPLACE FUNCTION catalog_commits.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).+(@.*$)', '$1***$2');

CREATE OR REPLACE FUNCTION catalog_commits.default.mask_ssn(ssn STRING)
RETURNS STRING
RETURN '***-**-****';

In [0]:
ALTER TABLE catalog.default.customers_masked
  ALTER COLUMN email SET MASK catalog.default.mask_email;

ALTER TABLE catalog.default.customers_masked
  ALTER COLUMN ssn SET MASK catalog.default.mask_ssn;


ALTER TABLE catalog_commits.default.customers_masked
  ALTER COLUMN email SET MASK catalog_commits.default.mask_email;

ALTER TABLE catalog_commits.default.customers_masked
  ALTER COLUMN ssn SET MASK catalog_commits.default.mask_ssn;

In [0]:
SELECT 'managed' AS table_type, *
FROM catalog.default.customers_masked
ORDER BY id;

In [0]:
SELECT 'catalog_commits' AS table_type, *
FROM catalog_commits.default.customers_masked
ORDER BY id;

In [0]:
CREATE OR REPLACE VIEW catalog.default.v_customers_masked AS
SELECT *
FROM catalog.default.customers_masked;

CREATE OR REPLACE VIEW catalog_commits.default.v_customers_masked AS
SELECT *
FROM catalog_commits.default.customers_masked;




SELECT 'catalog_commits_view' AS source, *
FROM catalog_commits.default.v_customers_masked
ORDER BY id;

In [0]:
SELECT 'managed_view' AS source, *
FROM catalog.default.v_customers_masked
ORDER BY id;

In [0]:
%python
from delta.tables import DeltaTable

spark.read.format("delta").load("abfss://unity@westus2uc.dfs.core.windows.net/catalog/__unitystorage/catalogs/b07fd208-8482-47d4-9ebc-85cab4293acc/tables/0d2459c8-2b3e-4fb5-a32f-1315563dbae7").show()